# 証券営業インテリジェンス ハンズオン
## Part 2: Cortex AI セキュリティ・アクセス制御

このノートブックでは、金融機関において特に重要な **Snowflake Cortex AI のセキュリティ設定**を学びます。
LLM の利用においてデータの安全性を確保するための具体的な設定・制御方法を体験します。

### このパートで学ぶこと

| 設定項目 | 概要 | 金融機関での重要性 |
|---|---|---|
| **クロスリージョン推論制御** | データを処理するリージョンを限定 | データ主権・規制遵守 |
| **Cortex アクセス制御 (RBAC)** | AI機能を使えるロールを制限 | 最小権限の原則 |
| **使用モデルの制限** | 利用可能なLLMモデルを指定 | 承認済みモデルのみ使用 |
| **AI 利用の監査** | いつ・誰が・何のモデルを使ったか記録 | コンプライアンス・ログ管理 |
| **データマスキング + AI** | PIIをマスキングしてからLLMへ送信 | 情報漏洩防止 |

### 🏦 金融機関特有の考慮事項

> **「AIを使うとデータが外部に出るのでは？」**
>
> Snowflake Cortex は、クエリを実行した Snowflake アカウントの
> インフラ内で処理されます（モデルは Snowflake が管理）。
> ただし、クロスリージョン設定によっては別リージョンのモデルを
> 使う場合があるため、本ノートブックの設定が重要です。

> ⚠️ **注意**: このノートブックの一部操作には `ACCOUNTADMIN` 権限が必要です。

> ⏱️ **このパートの目安時間: 20分**


In [ ]:
%%sql -r result_env
USE ROLE ACCOUNTADMIN;
USE DATABASE SNOWFINANCE_DB;
USE SCHEMA DEMO_SCHEMA;
USE WAREHOUSE DEMO_WH;
SELECT CURRENT_ROLE() AS ロール, CURRENT_REGION() AS リージョン,
       CURRENT_DATABASE() AS DB, CURRENT_WAREHOUSE() AS WH;

## 1. クロスリージョン推論の確認と設定

Snowflake Cortex の AI Functions を実行する際、LLM モデルが動作するリージョンを制御できます。

### リージョン設定の選択肢

| 設定値 | 意味 | 推奨シナリオ |
|---|---|---|
| `DISABLED` | 同一リージョンのモデルのみ使用 | **最高セキュリティ**（データ主権重視） |
| `ONLY_US` | 米国リージョンのモデルも使用可能 | US規制要件がある場合 |
| `ONLY_EU` | EUリージョンのモデルも使用可能 | GDPR 準拠 |
| `ANY_REGION` | 全リージョンのモデルを使用可能 | **最多モデル選択肢**（開発・PoC向け） |

> 💡 **日本法人の場合**: `DISABLED` に設定すると日本リージョンで
> 使用可能なモデルのみに限定されます。利用可能モデルが減りますが
> データが国外に出ません。ハンズオンでは `ANY_REGION` を使用します。

> ⏱️ **目安: 3分**


In [ ]:
%%sql -r result_cortex_params
-- 現在のCortex関連パラメータを確認
SHOW PARAMETERS LIKE 'CORTEX%' IN ACCOUNT;

In [ ]:
%%sql -r result_cross_region
-- クロスリージョン設定を確認（読み取り専用で確認）
SELECT
    KEY AS パラメータ名,
    VALUE AS 現在の設定値,
    DESCRIPTION AS 説明
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
WHERE KEY LIKE '%CROSS_REGION%' OR KEY LIKE '%CORTEX%';

### クロスリージョン設定の変更方法（参考）

実際に設定を変更する場合は以下のコマンドを使用します。
**ハンズオンでは変更しません**（`ANY_REGION` のまま進めます）。

```sql
-- ✅ 推奨（金融機関向け）: 同一リージョンのみ
ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'DISABLED';

-- 開発/PoC向け: 全リージョン許可（最多モデル選択肢）
ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'ANY_REGION';
```

> ⚠️ `DISABLED` にすると利用可能なモデルが制限されます。
> 本番環境に適用する前に、業務で使用するモデルが利用可能か確認してください。


## 2. Cortex アクセス制御（Model RBAC）

### SNOWFLAKE.CORTEX_USER データベースロール

Snowflake では `SNOWFLAKE.CORTEX_USER` データベースロールを持つロールのみが
Cortex AI Functions（`AI_COMPLETE`, `AI_CLASSIFY` 等）を実行できます。

**RBAC の仕組み:**
```
ACCOUNTADMIN / SYSADMIN
    └── SNOWFLAKE.CORTEX_USER を持つロール → AI Functions 実行可能 ✅
    └── SNOWFLAKE.CORTEX_USER を持たないロール → AI Functions 実行不可 ❌
```

> ⏱️ **目安: 8分**


In [ ]:
%%sql -r result_cortex_grants
-- 現在 CORTEX_USER を付与されているロールを確認
SHOW GRANTS OF DATABASE ROLE SNOWFLAKE.CORTEX_USER;

In [ ]:
%%sql -r result_create_restricted_role
-- デモ用の制限ロールを作成（Cortex 権限なし）
CREATE ROLE IF NOT EXISTS DEMO_NO_AI_ROLE;
GRANT USAGE ON DATABASE SNOWFINANCE_DB TO ROLE DEMO_NO_AI_ROLE;
GRANT USAGE ON SCHEMA DEMO_SCHEMA TO ROLE DEMO_NO_AI_ROLE;
GRANT SELECT ON ALL TABLES IN SCHEMA DEMO_SCHEMA TO ROLE DEMO_NO_AI_ROLE;
GRANT USAGE ON WAREHOUSE DEMO_WH TO ROLE DEMO_NO_AI_ROLE;
-- ※ SNOWFLAKE.CORTEX_USER は意図的に付与しない

SELECT 'DEMO_NO_AI_ROLE を作成しました（Cortex権限なし）' AS STATUS;

In [ ]:
%%sql -r result_role_grants
-- Cortex アクセスを付与する方法（参考）
-- ※ ハンズオン用ロールには付与済みのため、実際には実行不要

-- AI Functions を許可する場合:
-- GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE DEMO_NO_AI_ROLE;

-- AI Functions を禁止する場合（取り消し）:
-- REVOKE DATABASE ROLE SNOWFLAKE.CORTEX_USER FROM ROLE DEMO_NO_AI_ROLE;

-- 現在の SALES_ENGINEER ロールの Cortex 権限を確認
SHOW GRANTS TO ROLE SALES_ENGINEER;

## 3. 使用モデルの制御

### 承認済みモデルリスト

Snowflake では利用可能な Cortex LLM モデルをアカウント管理者が管理できます。
特定のモデルのみを業務利用として承認することで、未承認モデルへのアクセスを防ぎます。

### 現在利用可能なモデルの確認

> ⏱️ **目安: 3分**


In [ ]:
%%sql -r result_models
-- 利用可能な Cortex LLM モデルを一覧確認
SELECT
    MODEL_NAME AS モデル名,
    CREATED AS 追加日,
    COMMENT AS 説明
FROM SNOWFLAKE.INFORMATION_SCHEMA.CORTEX_AI_MODELS
ORDER BY MODEL_NAME;

### モデルの選定基準（金融機関向け）

| 考慮事項 | 推奨アプローチ |
|---|---|
| **データの機密性** | 顧客氏名・口座番号を含むクエリには小規模モデルを使用し、外部転送リスクを最小化 |
| **処理速度 vs 精度** | 一括処理（感情分析等）→ 小モデル、重要な提案生成 → 大モデル |
| **コスト管理** | `AI_SUMMARIZE` は自動選択、`AI_COMPLETE` はモデルを明示指定してコスト予測 |
| **モデルの所在** | クロスリージョン = `DISABLED` の場合、日本リージョンで使えるモデルのみ使用可能 |

```sql
-- 承認モデルのみ使用する例（コード規約として）
-- ✅ 承認済み小モデル（高速・低コスト）
SELECT SNOWFLAKE.CORTEX.COMPLETE('mistral-7b', '...') ...;

-- ✅ 承認済み大モデル（高精度・高コスト、重要用途のみ）  
SELECT SNOWFLAKE.CORTEX.COMPLETE('mistral-large2', '...') ...;

-- ❌ 未承認モデルは使わない（組織ポリシーで定義）
```


## 4. AI 利用の監査（Audit Log）

金融機関では「いつ・誰が・どのAIモデルを・何に使ったか」の記録が重要です。
Snowflake の `QUERY_HISTORY` ビューで Cortex AI の使用状況をトレースできます。

> ⏱️ **目安: 3分**


In [ ]:
%%sql -r result_audit_session
-- Cortex AI 使用状況の監査
-- ※ ACCOUNT_USAGE ビューはデータ反映まで最大45分かかるため
--   直近の利用はINFORMATION_SCHEMAで確認する
SELECT
    START_TIME AS 実行日時,
    USER_NAME AS 実行ユーザー,
    ROLE_NAME AS 実行ロール,
    LEFT(QUERY_TEXT, 100) AS クエリ（先頭100字）,
    EXECUTION_STATUS AS ステータス,
    TOTAL_ELAPSED_TIME / 1000 AS 実行時間秒
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY_BY_SESSION(
    RESULT_LIMIT => 20
))
WHERE (UPPER(QUERY_TEXT) LIKE '%SNOWFLAKE.CORTEX%'
    OR UPPER(QUERY_TEXT) LIKE '%AI_COMPLETE%'
    OR UPPER(QUERY_TEXT) LIKE '%AI_CLASSIFY%'
    OR UPPER(QUERY_TEXT) LIKE '%AI_SENTIMENT%'
    OR UPPER(QUERY_TEXT) LIKE '%AI_EXTRACT%')
  AND EXECUTION_STATUS = 'SUCCESS'
ORDER BY START_TIME DESC;

In [ ]:
%%sql -r result_audit_account
-- アカウント全体のCortex利用状況（過去7日間）
-- ※ ACCOUNTADMIN 権限が必要
SELECT
    DATE_TRUNC('day', START_TIME) AS 日付,
    USER_NAME AS ユーザー,
    ROLE_NAME AS ロール,
    COUNT(*) AS AI実行回数,
    COUNT(DISTINCT CASE WHEN QUERY_TEXT ILIKE '%mistral-large2%' THEN QUERY_ID END) AS 大モデル使用回数
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE START_TIME >= DATEADD('day', -7, CURRENT_TIMESTAMP())
  AND (QUERY_TEXT ILIKE '%SNOWFLAKE.CORTEX%'
    OR QUERY_TEXT ILIKE '%AI_COMPLETE%'
    OR QUERY_TEXT ILIKE '%AI_CLASSIFY%')
GROUP BY DATE_TRUNC('day', START_TIME), USER_NAME, ROLE_NAME
ORDER BY 日付 DESC, AI実行回数 DESC
LIMIT 20;

## 5. データマスキングポリシー × AI（多層防御）

**AI Functions に送る前にデータをマスキング**することで、
万が一の際もLLMに個人情報が渡らない仕組みを実現できます。

### 仕組み

```
ユーザーのクエリ
   ↓
[マスキングポリシー] ← ロール確認
   │ 管理者ロール   → 生データをAIへ送信
   │ 一般ロール     → AI_REDACT でPIIを除去してからAIへ送信
   ↓
AI_COMPLETE / AI_CLASSIFY 等
```

> ⏱️ **目安: 3分**


In [ ]:
%%sql -r result_masking_policy
-- マスキングポリシー: NOTES列を一般ロールからはPIIマスキングしてAIへ
CREATE OR REPLACE MASKING POLICY AI_SAFE_NOTES_POLICY
AS (val TEXT) RETURNS TEXT ->
    CASE
        -- 管理者・分析担当は生データを参照可能
        WHEN CURRENT_ROLE() IN ('ACCOUNTADMIN', 'SYSADMIN') THEN val
        -- それ以外はAI_REDACTで個人情報を除去してから表示
        ELSE SNOWFLAKE.CORTEX.AI_REDACT(val)::VARCHAR
    END;

-- ポリシーをDIM_CUSTOMERのNOTES列に適用
ALTER TABLE DIM_CUSTOMER
    MODIFY COLUMN NOTES SET MASKING POLICY AI_SAFE_NOTES_POLICY;

SELECT 'マスキングポリシー適用完了！' AS STATUS;

In [ ]:
%%sql -r result_masking_check
-- マスキング動作確認
-- 現在のロール（ACCOUNTADMIN）では生データが見える
SELECT CUSTOMER_ID, CUSTOMER_NAME, NOTES AS NOTES_現在のロール
FROM DIM_CUSTOMER
WHERE CUSTOMER_ID = 'C001';

### DEMO_NO_AI_ROLE でアクセスすると...

ACCOUNTADMIN でないロール（例: `DEMO_NO_AI_ROLE`）で同じクエリを実行すると、
NOTES 列が自動的に AI_REDACT でマスキングされた状態で返されます。

```sql
-- DEMO_NO_AI_ROLE で実行した場合のイメージ
USE ROLE DEMO_NO_AI_ROLE;
SELECT CUSTOMER_ID, NOTES FROM DIM_CUSTOMER WHERE CUSTOMER_ID = 'C001';
-- → NOTES: '***様から着信あり。住所：[REDACTED]。マイナンバー[REDACTED]にて...'
```

このポリシーにより、**AI に個人情報が渡るリスクを構造的に排除**できます。


In [ ]:
%%sql -r result_checklist
-- セキュリティ設定チェックリスト
SELECT
    'クロスリージョン設定' AS 設定項目,
    SYSTEM$GET_CURRENT_CONFIG('CORTEX_ENABLED_CROSS_REGION') AS 現在の値,
    '本番環境では DISABLED を推奨' AS 推奨
UNION ALL
SELECT
    'マスキングポリシー（NOTES）',
    (SELECT POLICY_NAME FROM INFORMATION_SCHEMA.POLICY_REFERENCES
     WHERE REF_COLUMN_NAME = 'NOTES' AND REF_ENTITY_NAME = 'DIM_CUSTOMER' LIMIT 1),
    'AI_SAFE_NOTES_POLICY が適用済みであること'
UNION ALL
SELECT
    'CORTEX_USER ロール付与状況',
    (SELECT LISTAGG(GRANTEE_NAME, ', ')
     FROM (SHOW GRANTS OF DATABASE ROLE SNOWFLAKE.CORTEX_USER) gs),
    '業務ロールのみに限定';

## まとめ

Part 2 完了！Cortex AI のセキュリティ設定を体験しました。

### 設定したセキュリティ機能

| 機能 | 設定内容 | 効果 |
|---|---|---|
| **クロスリージョン制御** | `CORTEX_ENABLED_CROSS_REGION` で管理 | データの処理リージョンを限定 |
| **Cortex RBAC** | `SNOWFLAKE.CORTEX_USER` ロールで管理 | 役割に応じてAI使用を許可/禁止 |
| **モデル管理** | 承認済みモデルリストをポリシーで管理 | 未承認LLMの使用防止 |
| **監査ログ** | `QUERY_HISTORY` でトレース | AI使用状況の完全な記録 |
| **多層マスキング** | マスキングポリシー + AI_REDACT | 構造的なPII保護 |

### 金融機関向けセキュリティベストプラクティス

```
1. データ主権: CORTEX_ENABLED_CROSS_REGION = 'DISABLED'（日本リージョン限定）
2. 最小権限: SNOWFLAKE.CORTEX_USER を業務ロールのみに付与
3. モデル統制: 承認済みモデルリストを組織ポリシーとして文書化
4. 監査: QUERY_HISTORY で月次レビュー
5. 多層防御: マスキングポリシー → AI_REDACT の二重保護
```

**次のステップ:** `03_ai_functions.ipynb` で Cortex AI Functions を体験してください。
